У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [25]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv('customer_segmentation_train.csv')

df = df.drop(columns='ID')

X = df.drop(columns='Segmentation')
y = df['Segmentation']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.copy()
X_test = X_test.copy()

numeric_cols = [
    'Age',
    'Work_Experience',
    'Family_Size'
]

categorical_cols = [
    'Gender',
    'Ever_Married',
    'Graduated',
    'Profession',
    'Spending_Score',
    'Var_1'
]

for col in numeric_cols:
    median = X_train[col].median()
    X_train[col] = X_train[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

for col in categorical_cols:
    mode = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode)
    X_test[col] = X_test[col].fillna(mode)

encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

cat_feature_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_cols
]

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [26]:
from imblearn.over_sampling import SMOTENC

smotenc = SMOTENC(
    categorical_features=cat_feature_indices,
    random_state=42
)

X_train_smote, y_train_smote = smotenc.fit_resample(
    X_train,
    y_train
)

In [27]:
from imblearn.combine import SMOTETomek

smotenc_tomek = SMOTENC(
    categorical_features=cat_feature_indices,
    random_state=42
)

smote_tomek = SMOTETomek(
    smote=smotenc_tomek,
    random_state=42
)

X_train_smote_tomek, y_train_smote_tomek = smote_tomek.fit_resample(
    X_train,
    y_train
)

In [28]:
print("До ресемплінгу:")
print(y_train.value_counts())

print("\nПісля SMOTENC:")
print(y_train_smote.value_counts())

print("\nПісля SMOTENC + Tomek:")
print(y_train_smote_tomek.value_counts())

До ресемплінгу:
Segmentation
D    1814
A    1578
C    1576
B    1486
Name: count, dtype: int64

Після SMOTENC:
Segmentation
A    1814
B    1814
C    1814
D    1814
Name: count, dtype: int64

Після SMOTENC + Tomek:
Segmentation
C    1551
D    1551
B    1511
A    1467
Name: count, dtype: int64


До ресемплінгу тренувальна вибірка була незбалансованою: клас D мав найбільше прикладів, а клас B — найменше. Після застосування SMOTENC кількість прикладів у всіх класах стала однаковою завдяки створенню синтетичних спостережень. Після SMOTENC + Tomek кількість прикладів дещо зменшилася, оскільки метод Tomek видалив неоднозначні спостереження на межі між класами, що може покращити якість навчання моделі.

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [29]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def train_and_evaluate(X_train, y_train, X_test, y_test, name):

    model = OneVsRestClassifier(
        LogisticRegression(max_iter=1000, random_state=42)
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print(classification_report(y_test, y_pred))

In [30]:
train_and_evaluate(
    X_train,
    y_train,
    X_test,
    y_test,
    "Оригінальні дані"
)

train_and_evaluate(
    X_train_smote,
    y_train_smote,
    X_test,
    y_test,
    "SMOTENC"
)

train_and_evaluate(
    X_train_smote_tomek,
    y_train_smote_tomek,
    X_test,
    y_test,
    "SMOTENC + Tomek"
)


Оригінальні дані
              precision    recall  f1-score   support

           A       0.39      0.39      0.39       394
           B       0.41      0.08      0.14       372
           C       0.47      0.64      0.54       394
           D       0.59      0.79      0.67       454

    accuracy                           0.49      1614
   macro avg       0.46      0.48      0.44      1614
weighted avg       0.47      0.49      0.45      1614


SMOTENC
              precision    recall  f1-score   support

           A       0.42      0.40      0.41       394
           B       0.37      0.14      0.21       372
           C       0.47      0.63      0.54       394
           D       0.62      0.78      0.69       454

    accuracy                           0.50      1614
   macro avg       0.47      0.49      0.46      1614
weighted avg       0.48      0.50      0.47      1614


SMOTENC + Tomek
              precision    recall  f1-score   support

           A       0.41      0.

Для порівняння моделей доцільно використати F1-score (macro avg), оскільки задача є багатокласовою, а класи спочатку були незбалансованими. Ця метрика однаково враховує всі класи та поєднує precision і recall.

Найкращі результати показали SMOTENC і SMOTENC + Tomek. Обидві моделі мають однаковий macro F1 = 0.46 та accuracy = 0.50, що трохи краще, ніж модель, навчена на оригінальних даних.

Різниця між моделями невелика, оскільки початковий дисбаланс класів був незначним. Ресемплінг трохи покращив якість класифікації, але не змінив її кардинально. Крім того, логістична регресія є досить стійкою до помірного дисбалансу класів.